In [1]:
import pandas as pd
import glob

files = sorted(
    glob.glob("/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/demand_extracted/*.[cC][sS][vV]")
)

print("Files found:", len(files))

all_rows = []

for f in files:
    with open(f, "r", errors="ignore") as file:
        for line in file:
            row = line.strip().split(",")
            all_rows.append(row)

print("Total rows loaded:", len(all_rows))

demand_raw = pd.DataFrame(all_rows)

print("Raw shape:", demand_raw.shape)
print(demand_raw.head(10))

Files found: 672
Total rows loaded: 5376
Raw shape: (5376, 10)
   0                   1                             2     3         4  \
0  C          NEMP.WORLD  ACTUAL_OPERATIONAL_DEMAND_HH  AEMO    PUBLIC   
1  I  OPERATIONAL_DEMAND                        ACTUAL     3  REGIONID   
2  D  OPERATIONAL_DEMAND                        ACTUAL     3      NSW1   
3  D  OPERATIONAL_DEMAND                        ACTUAL     3      QLD1   
4  D  OPERATIONAL_DEMAND                        ACTUAL     3       SA1   
5  D  OPERATIONAL_DEMAND                        ACTUAL     3      TAS1   
6  D  OPERATIONAL_DEMAND                        ACTUAL     3      VIC1   
7  C     "END OF REPORT"                             8  None      None   
8  C          NEMP.WORLD  ACTUAL_OPERATIONAL_DEMAND_HH  AEMO    PUBLIC   
9  I  OPERATIONAL_DEMAND                        ACTUAL     3  REGIONID   

                       5                   6                              7  \
0             2026/03/08            00:00:2

In [2]:
demand_raw[0] = demand_raw[0].astype(str).str.replace('"', '', regex=False).str.strip()

demand_df = demand_raw[demand_raw[0] == "D"].copy()

print("After D filter:", demand_df.shape)
print(demand_df.iloc[:10, :12])

After D filter: (3360, 10)
    0                   1       2  3     4                      5     6  7  8  \
2   D  OPERATIONAL_DEMAND  ACTUAL  3  NSW1  "2026/03/08 00:00:00"  7654  0  0   
3   D  OPERATIONAL_DEMAND  ACTUAL  3  QLD1  "2026/03/08 00:00:00"  6717  0  0   
4   D  OPERATIONAL_DEMAND  ACTUAL  3   SA1  "2026/03/08 00:00:00"  1279  0  0   
5   D  OPERATIONAL_DEMAND  ACTUAL  3  TAS1  "2026/03/08 00:00:00"   932  0  0   
6   D  OPERATIONAL_DEMAND  ACTUAL  3  VIC1  "2026/03/08 00:00:00"  4723  0  0   
10  D  OPERATIONAL_DEMAND  ACTUAL  3  NSW1  "2026/03/08 00:30:00"  7487  0  0   
11  D  OPERATIONAL_DEMAND  ACTUAL  3  QLD1  "2026/03/08 00:30:00"  6529  0  0   
12  D  OPERATIONAL_DEMAND  ACTUAL  3   SA1  "2026/03/08 00:30:00"  1300  0  0   
13  D  OPERATIONAL_DEMAND  ACTUAL  3  TAS1  "2026/03/08 00:30:00"   918  0  0   
14  D  OPERATIONAL_DEMAND  ACTUAL  3  VIC1  "2026/03/08 00:30:00"  4552  0  0   

                        9  
2   "2026/03/08 00:00:01"  
3   "2026/03/08 00:00:01"

In [3]:
demand_df = demand_df[[5, 4, 6]].copy()
demand_df.columns = ["SETTLEMENTDATE", "REGIONID", "TOTALDEMAND"]

In [4]:
demand_df["SETTLEMENTDATE"] = (
    demand_df["SETTLEMENTDATE"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

demand_df["REGIONID"] = (
    demand_df["REGIONID"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

demand_df["TOTALDEMAND"] = (
    demand_df["TOTALDEMAND"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

In [5]:
demand_df["SETTLEMENTDATE"] = pd.to_datetime(
    demand_df["SETTLEMENTDATE"],
    format="%Y/%m/%d %H:%M:%S",
    errors="coerce"
)

demand_df["TOTALDEMAND"] = pd.to_numeric(
    demand_df["TOTALDEMAND"],
    errors="coerce"
)

In [6]:
print("Regions:", demand_df["REGIONID"].unique())
print("Min date:", demand_df["SETTLEMENTDATE"].min())
print("Max date:", demand_df["SETTLEMENTDATE"].max())
print("Null dates:", demand_df["SETTLEMENTDATE"].isna().sum())

Regions: ['NSW1' 'QLD1' 'SA1' 'TAS1' 'VIC1']
Min date: 2026-03-08 00:00:00
Max date: 2026-03-21 23:30:00
Null dates: 0


In [7]:
demand_df = demand_df[demand_df["REGIONID"] == "VIC1"].copy()

demand_df = demand_df[
    (demand_df["SETTLEMENTDATE"] >= "2026-03-09") &
    (demand_df["SETTLEMENTDATE"] < "2026-03-17")
].copy()

In [8]:
demand_df = demand_df.drop_duplicates(subset=["SETTLEMENTDATE", "REGIONID"])
demand_df = demand_df.sort_values("SETTLEMENTDATE").reset_index(drop=True)

print("Final shape:", demand_df.shape)
print(demand_df.head())
print(demand_df.tail())

Final shape: (384, 3)
       SETTLEMENTDATE REGIONID  TOTALDEMAND
0 2026-03-09 00:00:00     VIC1         4681
1 2026-03-09 00:30:00     VIC1         4490
2 2026-03-09 01:00:00     VIC1         4392
3 2026-03-09 01:30:00     VIC1         4302
4 2026-03-09 02:00:00     VIC1         4211
         SETTLEMENTDATE REGIONID  TOTALDEMAND
379 2026-03-16 21:30:00     VIC1         5283
380 2026-03-16 22:00:00     VIC1         5088
381 2026-03-16 22:30:00     VIC1         4942
382 2026-03-16 23:00:00     VIC1         4878
383 2026-03-16 23:30:00     VIC1         4974


In [9]:
demand_df = demand_df.drop_duplicates(subset=["SETTLEMENTDATE", "REGIONID"])
demand_df = demand_df.sort_values("SETTLEMENTDATE").reset_index(drop=True)

print("Final shape:", demand_df.shape)
print(demand_df.head())
print(demand_df.tail())

Final shape: (384, 3)
       SETTLEMENTDATE REGIONID  TOTALDEMAND
0 2026-03-09 00:00:00     VIC1         4681
1 2026-03-09 00:30:00     VIC1         4490
2 2026-03-09 01:00:00     VIC1         4392
3 2026-03-09 01:30:00     VIC1         4302
4 2026-03-09 02:00:00     VIC1         4211
         SETTLEMENTDATE REGIONID  TOTALDEMAND
379 2026-03-16 21:30:00     VIC1         5283
380 2026-03-16 22:00:00     VIC1         5088
381 2026-03-16 22:30:00     VIC1         4942
382 2026-03-16 23:00:00     VIC1         4878
383 2026-03-16 23:30:00     VIC1         4974


In [10]:
demand_df.to_csv(
    "/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/processed/demand_clean.csv",
    index=False
)

print("✅ Demand saved successfully")

✅ Demand saved successfully
